In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from scipy.stats import gaussian_kde
import warnings, os
warnings.filterwarnings('ignore')

In [2]:
BASE_PATH = '../../outputs'
EDGES_PATH  = f'{BASE_PATH}/final/knowledge_edges.csv'
MATRIX_PATH = f'{BASE_PATH}/final/novelty_feature_matrix_with_score.csv'
PAPERS_PATH = f'{BASE_PATH}/final/paper_nodes.csv'
EMB_PATH    = f'{BASE_PATH}/final/abstract_embeddings.npy'
IDS_PATH    = f'{BASE_PATH}/final/paper_ids.npy'
OUT_DIR     = f'{BASE_PATH}/visuals'
os.makedirs(OUT_DIR, exist_ok=True)

In [3]:
COLOURS = {
    'NOVEL': '#2ecc71',   # green
    'SKG':   '#e74c3c',   # red
    'BLOG':  '#3498db'    # blue
}
SPLIT_ORDER = ['NOVEL', 'SKG', 'BLOG']

In [5]:
print("Loading data...")
matrix = pd.read_csv(MATRIX_PATH)
papers = pd.read_csv(PAPERS_PATH)
emb    = np.load(EMB_PATH)
ids    = np.load(IDS_PATH, allow_pickle=True)

# Merge split + domain + title into matrix
matrix = matrix.merge(
    papers[['node_id', 'split', 'domain', 'title']],
    left_on='paper_id', right_on='node_id', how='left'
)

print(f"Matrix: {matrix.shape}")
print(f"Splits: {matrix['split'].value_counts().to_dict()}")

id2idx = {pid: i for i, pid in enumerate(ids)}

Loading data...
Matrix: (4242, 18)
Splits: {'SKG': 2916, 'NOVEL': 762, 'BLOG': 564}


In [6]:
print("\n[1/6] t-SNE of SciBERT embeddings...")

# Map matrix papers to their embeddings
matrix['emb_idx'] = matrix['paper_id'].map(id2idx)
has_emb = matrix['emb_idx'].notna()
emb_sub = emb[matrix.loc[has_emb, 'emb_idx'].astype(int).values]
meta_sub = matrix[has_emb].reset_index(drop=True)

# PCA to 50d first (speeds up t-SNE significantly)
print("  PCA 768 → 50d...")
pca50 = PCA(n_components=50, random_state=42)
emb50 = pca50.fit_transform(emb_sub)

print("  t-SNE 50 → 2d (this takes ~2 min)...")
tsne = TSNE(n_components=2, perplexity=40, max_iter=1000,
            random_state=42, learning_rate='auto', init='pca')
emb2d = tsne.fit_transform(emb50)

meta_sub['tsne_x'] = emb2d[:, 0]
meta_sub['tsne_y'] = emb2d[:, 1]

fig, ax = plt.subplots(figsize=(10, 8))

for split in SPLIT_ORDER:
    sub = meta_sub[meta_sub['split'] == split]
    ax.scatter(sub['tsne_x'], sub['tsne_y'],
               c=COLOURS[split], label=split,
               alpha=0.5, s=8, linewidths=0,
               rasterized=True)

ax.set_title('t-SNE of SciBERT Paper Embeddings\nColoured by Split',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('t-SNE Dimension 1', fontsize=11)
ax.set_ylabel('t-SNE Dimension 2', fontsize=11)
ax.legend(title='Split', fontsize=10, title_fontsize=10,
          markerscale=3, framealpha=0.9)
ax.tick_params(labelsize=9)
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/01_tsne_embeddings.png', dpi=150, bbox_inches='tight')
plt.close()
print(f"  Saved → {OUT_DIR}/01_tsne_embeddings.png")

# Save t-SNE coords for potential reuse
meta_sub[['paper_id','split','domain','tsne_x','tsne_y']].to_csv(
    f'{OUT_DIR}/tsne_coords.csv', index=False)


[1/6] t-SNE of SciBERT embeddings...
  PCA 768 → 50d...
  t-SNE 50 → 2d (this takes ~2 min)...
  Saved → ../../outputs/visuals/01_tsne_embeddings.png


In [7]:
print("\n[2/6] Novelty score distribution (KDE + boxplot)...")

fig = plt.figure(figsize=(12, 9))
gs  = GridSpec(2, 1, height_ratios=[3, 1], hspace=0.08)
ax_kde = fig.add_subplot(gs[0])
ax_box = fig.add_subplot(gs[1], sharex=ax_kde)

scores_by_split = {
    split: matrix.loc[matrix['split'] == split, 'final_novelty_score'].dropna().values
    for split in SPLIT_ORDER
}

# KDE curves
x_range = np.linspace(
    matrix['final_novelty_score'].min() - 0.02,
    matrix['final_novelty_score'].max() + 0.02,
    500
)

for split in SPLIT_ORDER:
    vals = scores_by_split[split]
    if len(vals) > 5:
        kde = gaussian_kde(vals, bw_method=0.3)
        ax_kde.plot(x_range, kde(x_range),
                    color=COLOURS[split], linewidth=2.5, label=split)
        ax_kde.fill_between(x_range, kde(x_range),
                            alpha=0.15, color=COLOURS[split])
        ax_kde.axvline(vals.mean(), color=COLOURS[split],
                       linestyle='--', alpha=0.8, linewidth=1.5)

ax_kde.set_ylabel('Density', fontsize=11)
ax_kde.set_title('Final Novelty Score Distribution by Split\n'
                 '(dashed lines = group means)',
                 fontsize=14, fontweight='bold', pad=12)
ax_kde.legend(title='Split', fontsize=10, title_fontsize=10)
ax_kde.grid(True, alpha=0.2)
plt.setp(ax_kde.get_xticklabels(), visible=False)

# Boxplot
bp_data   = [scores_by_split[s] for s in SPLIT_ORDER]
bp_colors = [COLOURS[s] for s in SPLIT_ORDER]
bp = ax_box.boxplot(bp_data, vert=False, patch_artist=True,
                    labels=SPLIT_ORDER, widths=0.5,
                    medianprops=dict(color='black', linewidth=2))
for patch, color in zip(bp['boxes'], bp_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax_box.set_xlabel('Final Novelty Score', fontsize=11)
ax_box.grid(True, alpha=0.2, axis='x')

# Annotate means
for i, (split, vals) in enumerate(scores_by_split.items(), 1):
    ax_box.axvline(vals.mean(), color=COLOURS[split],
                   linestyle='--', alpha=0.6, linewidth=1.2)

plt.savefig(f'{OUT_DIR}/02_score_distribution.png', dpi=150, bbox_inches='tight')
plt.close()
print(f"  Saved → {OUT_DIR}/02_score_distribution.png")


[2/6] Novelty score distribution (KDE + boxplot)...
  Saved → ../../outputs/visuals/02_score_distribution.png


In [8]:
print("\n[3/6] Temporal novelty trend...")

yr_stats = (matrix[matrix['split'].isin(['NOVEL','SKG'])]
            .groupby(['year','split'])['final_novelty_score']
            .agg(['mean','sem','count'])
            .reset_index())

fig, ax = plt.subplots(figsize=(12, 6))

for split in ['NOVEL', 'SKG']:
    sub = yr_stats[yr_stats['split'] == split].sort_values('year')
    # Only show years with >= 3 papers
    sub = sub[sub['count'] >= 3]

    ax.plot(sub['year'], sub['mean'],
            color=COLOURS[split], linewidth=2.5,
            marker='o', markersize=6, label=split)
    ax.fill_between(sub['year'],
                    sub['mean'] - sub['sem'],
                    sub['mean'] + sub['sem'],
                    alpha=0.2, color=COLOURS[split])

    # Annotate n per year
    for _, row in sub.iterrows():
        ax.annotate(f"n={int(row['count'])}",
                    xy=(row['year'], row['mean']),
                    xytext=(0, 9), textcoords='offset points',
                    fontsize=6.5, ha='center',
                    color=COLOURS[split], alpha=0.8)

ax.set_xlabel('Publication Year', fontsize=11)
ax.set_ylabel('Mean Final Novelty Score', fontsize=11)
ax.set_title('Temporal Novelty Trend\nMean Score per Year (±SEM, n≥3)',
             fontsize=14, fontweight='bold', pad=12)
ax.legend(title='Split', fontsize=10, title_fontsize=10)
ax.grid(True, alpha=0.2)
ax.set_xlim(2009.5, 2025.5)
ax.xaxis.set_major_locator(plt.MultipleLocator(2))

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/03_temporal_trend.png', dpi=150, bbox_inches='tight')
plt.close()
print(f"  Saved → {OUT_DIR}/03_temporal_trend.png")


[3/6] Temporal novelty trend...
  Saved → ../../outputs/visuals/03_temporal_trend.png


In [9]:
print("\n[4/6] Domain comparison bar chart...")

domain_stats = (matrix[matrix['split'].isin(['NOVEL','SKG'])]
                .groupby(['domain','split'])['final_novelty_score']
                .agg(['mean','sem','count'])
                .reset_index())

domains = sorted(matrix['domain'].dropna().unique())
x       = np.arange(len(domains))
width   = 0.35

fig, ax = plt.subplots(figsize=(12, 6))

for i, split in enumerate(['NOVEL', 'SKG']):
    sub = domain_stats[domain_stats['split'] == split].set_index('domain')
    means = [sub.loc[d, 'mean'] if d in sub.index else 0 for d in domains]
    sems  = [sub.loc[d, 'sem']  if d in sub.index else 0 for d in domains]
    counts= [int(sub.loc[d, 'count']) if d in sub.index else 0 for d in domains]

    bars = ax.bar(x + i * width - width/2, means,
                  width, label=split,
                  color=COLOURS[split], alpha=0.8,
                  yerr=sems, capsize=4,
                  error_kw=dict(elinewidth=1.5, ecolor='black', alpha=0.6))

    # Annotate n
    for bar, n in zip(bars, counts):
        if n > 0:
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + 0.005,
                    f'n={n}', ha='center', va='bottom',
                    fontsize=7, color='black', alpha=0.8)

ax.set_xlabel('Domain', fontsize=11)
ax.set_ylabel('Mean Final Novelty Score', fontsize=11)
ax.set_title('Domain Comparison: NOVEL vs SKG\nMean Final Novelty Score (±SEM)',
             fontsize=14, fontweight='bold', pad=12)
ax.set_xticks(x)
ax.set_xticklabels(domains, fontsize=10)
ax.legend(title='Split', fontsize=10, title_fontsize=10)
ax.grid(True, alpha=0.2, axis='y')
ax.set_ylim(0, ax.get_ylim()[1] * 1.1)

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/04_domain_comparison.png', dpi=150, bbox_inches='tight')
plt.close()
print(f"  Saved → {OUT_DIR}/04_domain_comparison.png")



[4/6] Domain comparison bar chart...
  Saved → ../../outputs/visuals/04_domain_comparison.png


In [10]:
print("\n[5/6] TransE vs Semantic scatter...")

# Only papers that have both scores
has_both = matrix['transE_score'].notna()
scatter_df = matrix[has_both].copy()

# Invert transE for display (high = novel)
max_t = scatter_df['transE_score'].max()
scatter_df['transE_novelty'] = max_t - scatter_df['transE_score']
# Semantic novelty: lower similarity = more novel
scatter_df['sem_novelty'] = 1 - scatter_df['semantic_knn']

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=False)

for ax, split in zip(axes, SPLIT_ORDER):
    sub = scatter_df[scatter_df['split'] == split]
    if len(sub) == 0:
        ax.text(0.5, 0.5, f'No {split} data',
                ha='center', va='center', transform=ax.transAxes)
        continue

    ax.scatter(sub['sem_novelty'], sub['transE_novelty'],
               c=COLOURS[split], alpha=0.4, s=18,
               linewidths=0, rasterized=True)

    # Correlation line
    if len(sub) > 5:
        x_vals = sub['sem_novelty'].values
        y_vals = sub['transE_novelty'].values
        corr   = np.corrcoef(x_vals, y_vals)[0, 1]
        m, b   = np.polyfit(x_vals, y_vals, 1)
        x_line = np.linspace(x_vals.min(), x_vals.max(), 100)
        ax.plot(x_line, m * x_line + b,
                color='black', linewidth=1.5, alpha=0.7,
                linestyle='--')
        ax.text(0.05, 0.95, f'r = {corr:.3f}\nn = {len(sub)}',
                transform=ax.transAxes, fontsize=9,
                va='top', bbox=dict(boxstyle='round', alpha=0.3))

    ax.set_title(split, fontsize=12, fontweight='bold', color=COLOURS[split])
    ax.set_xlabel('Semantic Novelty\n(1 − cosine similarity)', fontsize=9)
    ax.set_ylabel('TransE Novelty\n(inverted distance)', fontsize=9)
    ax.grid(True, alpha=0.2)

fig.suptitle('TransE Score vs Semantic Score\nby Split',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/05_transE_vs_semantic.png', dpi=150, bbox_inches='tight')
plt.close()
print(f"  Saved → {OUT_DIR}/05_transE_vs_semantic.png")


[5/6] TransE vs Semantic scatter...
  Saved → ../../outputs/visuals/05_transE_vs_semantic.png


In [11]:
print("\n[6/6] Top / bottom papers table...")

N_SHOW = 10

novel_df = matrix[matrix['split'] == 'NOVEL'].copy()
novel_df['title_short'] = novel_df['title'].fillna('(no title)').str[:60]

top_papers = novel_df.nlargest(N_SHOW, 'final_novelty_score')[
    ['paper_id','title_short','domain','year','final_novelty_score',
     'transE_score','semantic_knn']
].reset_index(drop=True)

bot_papers = novel_df.nsmallest(N_SHOW, 'final_novelty_score')[
    ['paper_id','title_short','domain','year','final_novelty_score',
     'transE_score','semantic_knn']
].reset_index(drop=True)

def make_table_fig(df, title, row_colour, n_cols=6):
    col_labels = ['Paper ID','Title (truncated)','Domain',
                  'Year','Novelty Score','TransE Score']
    cell_data = []
    for _, row in df.iterrows():
        cell_data.append([
            row['paper_id'],
            row['title_short'] + ('…' if len(str(row.get('title_short',''))) >= 60 else ''),
            row['domain'],
            str(int(row['year'])) if not pd.isna(row['year']) else '?',
            f"{row['final_novelty_score']:.4f}",
            f"{row['transE_score']:.3f}" if not pd.isna(row['transE_score']) else 'N/A'
        ])

    fig, ax = plt.subplots(figsize=(16, 0.55 * (len(df) + 2)))
    ax.axis('off')

    tbl = ax.table(
        cellText=cell_data,
        colLabels=col_labels,
        loc='center',
        cellLoc='left'
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(8.5)
    tbl.auto_set_column_width(col=list(range(n_cols)))

    # Header style
    for j in range(n_cols):
        tbl[0, j].set_facecolor('#2c3e50')
        tbl[0, j].set_text_props(color='white', fontweight='bold')

    # Row alternating colours
    for i in range(1, len(df) + 1):
        for j in range(n_cols):
            if i % 2 == 0:
                tbl[i, j].set_facecolor('#f8f9fa')
            else:
                tbl[i, j].set_facecolor(row_colour)

    ax.set_title(title, fontsize=13, fontweight='bold',
                 pad=20, loc='left')
    plt.tight_layout()
    return fig

fig_top = make_table_fig(
    top_papers,
    f'Top {N_SHOW} Most Novel Papers (NOVEL split)',
    '#d5f5e3'
)
fig_top.savefig(f'{OUT_DIR}/06a_top_papers.png', dpi=150, bbox_inches='tight')
plt.close()

fig_bot = make_table_fig(
    bot_papers,
    f'Bottom {N_SHOW} Least Novel Papers (NOVEL split)',
    '#fadbd8'
)
fig_bot.savefig(f'{OUT_DIR}/06b_bottom_papers.png', dpi=150, bbox_inches='tight')
plt.close()

print(f"  Saved → {OUT_DIR}/06a_top_papers.png")
print(f"  Saved → {OUT_DIR}/06b_bottom_papers.png")


[6/6] Top / bottom papers table...
  Saved → ../../outputs/visuals/06a_top_papers.png
  Saved → ../../outputs/visuals/06b_bottom_papers.png


## Temporal Modeling Results (Member 3)

Visualisations from the TGNN temporal evolution pipeline.

In [12]:
print("\n[7/10] KG Snapshot Growth Curve...")

snapshot = pd.read_csv(f'{BASE_PATH}/final/kg_snapshot_stats.csv')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Cumulative edge growth
ax1.fill_between(snapshot['year'], snapshot['cumulative_edges'], alpha=0.3, color='#3498db')
ax1.plot(snapshot['year'], snapshot['cumulative_edges'], color='#3498db', linewidth=2, marker='o', markersize=4)
ax1.set_xlabel('Year')
ax1.set_ylabel('Cumulative Unique Triples')
ax1.set_title('KG Growth: Cumulative Triples Over Time')
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e3:.0f}K'))
ax1.grid(axis='y', alpha=0.3)

# Novel triples added per year
ax2.bar(snapshot['year'], snapshot['edges_added'], color='#e74c3c', alpha=0.7, label='Edges Added')
ax2.bar(snapshot['year'], snapshot['novel_triples'], color='#2ecc71', alpha=0.7, label='Novel Triples')
ax2.set_xlabel('Year')
ax2.set_ylabel('Count')
ax2.set_title('Edge Addition Dynamics per Year')
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e3:.0f}K'))
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/07_kg_growth.png', dpi=150, bbox_inches='tight')
plt.close()
print("  Saved 07_kg_growth.png")


[7/10] KG Snapshot Growth Curve...
  Saved 07_kg_growth.png


In [13]:
print("\n[8/10] Emerging Concept Detection...")

emerging = pd.read_csv(f'{BASE_PATH}/final/emerging_concepts.csv')
top_n = emerging.head(15).copy()
# Truncate long names
top_n['label'] = top_n['name'].str[:40]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(top_n['label'], top_n['growth_ratio'], color='#9b59b6', alpha=0.8)
ax.set_xlabel('Growth Ratio (late / early degree)')
ax.set_title('Top 15 Emerging Concepts (2015–2020)')
ax.axvline(x=1.0, color='red', linestyle='--', alpha=0.5, label='No growth')
ax.invert_yaxis()
ax.legend()
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/08_emerging_concepts.png', dpi=150, bbox_inches='tight')
plt.close()
print("  Saved 08_emerging_concepts.png")


[8/10] Emerging Concept Detection...
  Saved 08_emerging_concepts.png


In [14]:
print("\n[9/10] Disruption vs Incremental Classification...")

disrupt = pd.read_csv(f'{BASE_PATH}/final/disruption_classification.csv')
disrupt = disrupt[disrupt['split'].isin(['SKG','NOVEL'])]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Disruption index distribution by split
for sp, col in [('SKG', '#e74c3c'), ('NOVEL', '#2ecc71')]:
    sub = disrupt[disrupt['split']==sp]['disruption_index']
    axes[0].hist(sub, bins=40, alpha=0.6, color=col, label=sp, density=True)
axes[0].set_xlabel('Disruption Index')
axes[0].set_ylabel('Density')
axes[0].set_title('Disruption Index Distribution by Split')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Right: Stacked bar chart — Disruptive vs Incremental per split
ct = disrupt.groupby(['split','contribution_type']).size().unstack(fill_value=0)
ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
ct_pct.plot(kind='bar', stacked=True, ax=axes[1],
            color=['#e74c3c', '#3498db'], alpha=0.8)
axes[1].set_xlabel('Split')
axes[1].set_ylabel('Percentage (%)')
axes[1].set_title('Contribution Type by Split')
axes[1].legend(title='Type')
axes[1].tick_params(axis='x', rotation=0)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/09_disruption_classification.png', dpi=150, bbox_inches='tight')
plt.close()
print("  Saved 09_disruption_classification.png")


[9/10] Disruption vs Incremental Classification...
  Saved 09_disruption_classification.png


In [15]:
print("\n[10/10] TGNN Novelty Trend vs Year...")

tgnn_ts = pd.read_csv(f'{BASE_PATH}/final/tgnn_timeseries.csv')

fig, ax = plt.subplots(figsize=(12, 5))

for sp in ['SKG', 'NOVEL']:
    sub = tgnn_ts[tgnn_ts['split']==sp]
    if len(sub) == 0:
        continue
    col = COLOURS.get(sp, 'grey')
    ax.plot(sub['year'], sub['tgnn_mean'], marker='o', label=sp, color=col, linewidth=2)
    ax.fill_between(sub['year'],
                    sub['tgnn_mean'] - sub['tgnn_std'].fillna(0),
                    sub['tgnn_mean'] + sub['tgnn_std'].fillna(0),
                    alpha=0.15, color=col)

ax.set_xlabel('Year')
ax.set_ylabel('TGNN Novelty Score (mean ± std)')
ax.set_title('Temporal GNN Novelty Score Evolution')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/10_tgnn_novelty_trend.png', dpi=150, bbox_inches='tight')
plt.close()
print("  Saved 10_tgnn_novelty_trend.png")


[10/10] TGNN Novelty Trend vs Year...
  Saved 10_tgnn_novelty_trend.png


In [16]:
print(f"\n{'='*55}")
print(f"All 10 visualisations saved to ./{OUT_DIR}/")
print(f"{'='*55}")
files = [
    '01_tsne_embeddings.png   — SciBERT clusters by split',
    '02_score_distribution.png — KDE + boxplot',
    '03_temporal_trend.png    — Mean score per year',
    '04_domain_comparison.png — NOVEL vs SKG per domain',
    '05_transE_vs_semantic.png— Signal correlation scatter',
    '06a_top_papers.png       — Most novel papers table',
    '06b_bottom_papers.png    — Least novel papers table',
    'tsne_coords.csv          — t-SNE coordinates (reusable)',
    '07_kg_growth.png         — KG cumulative edge growth curve',
    '08_emerging_concepts.png — Top emerging research concepts',
    '09_disruption_classification.png — Disruptive vs incremental contributions',
    '10_tgnn_novelty_trend.png— TGNN novelty score evolution over time',
]
for f in files:
    print(f'  {f}')

# Print top/bottom paper lists to console as well
from scipy.stats import mannwhitneyu
nov = matrix.loc[matrix['split']=='NOVEL','final_novelty_score'].values
skg = matrix.loc[matrix['split']=='SKG','final_novelty_score'].values
st,p = mannwhitneyu(nov,skg,alternative='greater')
r = st/(len(nov)*len(skg))

print(f"\n── Final Score Summary ──")
print(f"NOVEL: mean={nov.mean():.4f}  std={nov.std():.4f}  n={len(nov)}")
print(f"SKG:   mean={skg.mean():.4f}  std={skg.std():.4f}  n={len(skg)}")
print(f"Mann-Whitney: p={p:.8f}  r={r:.4f}  ✅")

print(f"\n── Top 5 Most Novel Papers ──")
print(top_papers[['paper_id','domain','year','final_novelty_score']]
      .head(5).to_string(index=False))

print(f"\n── Bottom 5 Least Novel Papers ──")
print(bot_papers[['paper_id','domain','year','final_novelty_score']]
      .head(5).to_string(index=False))


All 10 visualisations saved to ./../../outputs/visuals/
  01_tsne_embeddings.png   — SciBERT clusters by split
  02_score_distribution.png — KDE + boxplot
  03_temporal_trend.png    — Mean score per year
  04_domain_comparison.png — NOVEL vs SKG per domain
  05_transE_vs_semantic.png— Signal correlation scatter
  06a_top_papers.png       — Most novel papers table
  06b_bottom_papers.png    — Least novel papers table
  tsne_coords.csv          — t-SNE coordinates (reusable)
  07_kg_growth.png         — KG cumulative edge growth curve
  08_emerging_concepts.png — Top emerging research concepts
  09_disruption_classification.png — Disruptive vs incremental contributions
  10_tgnn_novelty_trend.png— TGNN novelty score evolution over time

── Final Score Summary ──
NOVEL: mean=0.3764  std=0.0777  n=762
SKG:   mean=0.3518  std=0.0722  n=2916
Mann-Whitney: p=0.00000000  r=0.6005  ✅

── Top 5 Most Novel Papers ──
             paper_id domain  year  final_novelty_score
 NOVEL_SA_W4295795036   